Переходим к другой вехе, классификации.

In [1]:
!pip install catboost

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

In [3]:
df = pd.read_pickle('/content/dataset_after_eda.pkl')
df = df.drop_duplicates()
df.shape

(958, 211)

In [4]:
y = df['IC50_above_median'].astype(int)
X = df.drop(['IC50, mM', 'CC50, mM', 'SI', 'IC50_above_median', 'CC50_above_median', 'SI_above_median', 'SI_above_8'], axis=1)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape[0])
print(X_test.shape[0])

766
192


Баланс классов для IC50 у нас получился 50/50 (из EDA), соответсвенно можно использовать accuracy

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report)


models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("regressor", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0, allow_writing_files=False)
}

baseline_results = []

for name, model in models.items():
    # пайплайн сам отмасштабирует, мы указали
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    baseline_results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1-Score": round(f1_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_proba), 4)
    })

df_baseline = pd.DataFrame(baseline_results).sort_values(by="ROC-AUC", ascending=False)
df_baseline

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
3,CatBoost,0.7448,0.7320,0.7553,0.7435,0.8124
2,Gradient Boosting,0.7188,0.7128,0.7128,0.7128,0.8094
1,Random Forest,0.7344,0.7172,0.7553,0.7358,0.7807
0,Logistic Regression,0.7031,0.6907,0.7128,0.7016,0.7771


По умолчанию CatBoost и Gradient Boosting показали лучший ROC AUC = 0.81, Random Forest чуть отстал = 0.78. После настройки гиперпараметров ситуация изменилась, RF вышел на первое место = 0.8113, а CatBoost и GB показали идентичные результаты = 0.807.

In [7]:
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, None],
    'min_samples_split': [2, 5]
}

gb_param_grid = {
    'n_estimators': [100, 150],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 4, 5]
}

cat_param_grid = {
    'iterations': [300, 500],
    'learning_rate': [0.03, 0.05],
    'depth': [4, 6]
}

cv_rf = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
                     rf_param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)

cv_gb = GridSearchCV(GradientBoostingClassifier(random_state=42),
                     gb_param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)

cv_cat = GridSearchCV(CatBoostClassifier(random_state=42, verbose=0, allow_writing_files=False),
                      cat_param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)

print("Random Forest")
cv_rf.fit(X_train, y_train)

print("Gradient Boosting")
cv_gb.fit(X_train, y_train)

print("CatBoost")
cv_cat.fit(X_train, y_train)

best_models = {
    "Random Forest": cv_rf.best_estimator_,
    "Gradient Boosting": cv_gb.best_estimator_,
    "CatBoost": cv_cat.best_estimator_
}

final_results = []

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    final_results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc_auc, 4)
    })


    print("Матрица ошибок для", name)
    print(confusion_matrix(y_test, y_pred))

df_final = pd.DataFrame(final_results).sort_values(by="ROC-AUC", ascending=False)
df_final

Random Forest
Fitting 3 folds for each of 18 candidates, totalling 54 fits
Gradient Boosting
Fitting 3 folds for each of 18 candidates, totalling 54 fits
CatBoost
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Матрица ошибок для Random Forest
[[73 25]
 [21 73]]
Матрица ошибок для Gradient Boosting
[[73 25]
 [27 67]]
Матрица ошибок для CatBoost
[[73 25]
 [27 67]]


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Random Forest,0.7604,0.7449,0.7766,0.7604,0.8113
1,Gradient Boosting,0.7292,0.7283,0.7128,0.7204,0.8070
2,CatBoost,0.7292,0.7283,0.7128,0.7204,0.8060


#Вывод
Ориентируемся на roc-auc, Random Forest лучший результат и финальная модель, ROC AUC = 0.8113, Accuracy = 0.7604. Модель продемонстрировала высокую способность к обнаружению соединений с IC50 выше медианного значения, Recall = 0.7766, опередив градиентный бустинг. Логистическая регрессия уступает ансамблям, ничего не изменилось. Ну это очевидно, что чуда бы не произошло, логрег исследует линейную разделимость классов.